In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("sms.csv")

In [3]:
data.head(5)

,label,text
0,ham,Hey RAV!! Are we still meeting @ 6pm today??? ...
1,spam,CONGRATULATIONS!!! You have WON a FREE iPhone ...
2,ham,"Rose, can you pls send me the project report b..."
3,spam,URGENT!!! Your bank account is suspended!!! Ve...
4,ham,Hey!!! I'm running late today :( Traffic is cr...


In [4]:
import spacy

In [5]:
!python3 -m spacy download en_core_web_lg


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 9.3 MB/s  0:00:400:00:0100:02

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


In [6]:
nlp = spacy.load("en_core_web_lg")
def dataprocessing(text):
    doc = nlp(text)
    tokens = []
    for token in doc:
        if token.is_stop:
            continue
        if token.is_punct:
            continue
        tokens.append(token.lemma_.lower())
    return " ".join(tokens)
data['text'] = data["text"].apply(dataprocessing)
        
        

In [7]:
data.head(5)

,label,text
0,ham,hey rav meet 6 pm today bring document laptop ...
1,spam,congratulation win free iphone 15 worth $ 999 ...
2,ham,rose pls send project report 5 pm need review ...
3,spam,urgent bank account suspend verify immediately...
4,ham,hey run late today traffic crazyyyy reach offi...


In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(data['text'], data['label'], test_size=0.2, random_state=42)

In [9]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data['label'] = le.fit_transform(data['label'])

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [11]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)

In [12]:
vectorizer.get_feature_names_out()

array(['000', '07045612389', '08098765432', '09012345678', '10', '15',
       '20', '200', '30am', '5000', '80090', '999', 'account', 'act',
       'afternoon', 'alert', 'amazon', 'amazongiftbonus', 'appreciate',
       'area', 'asap', 'assignment', 'atm', 'available', 'avoid',
       'awesome', 'bank', 'block', 'bring', 'bro', 'bug', 'buy', 'cable',
       'card', 'cash', 'catch', 'charger', 'chat', 'check', 'claim',
       'class', 'clean', 'click', 'code', 'coffee', 'com',
       'congratulation', 'consultation', 'crazyyyy', 'customer',
       'dataset', 'datingnow', 'datum', 'deactivate', 'deal', 'dear',
       'detail', 'dial', 'document', 'email', 'evening', 'exclusive',
       'expire', 'fi', 'final', 'forget', 'free', 'freegift', 'gift',
       'github', 'gym', 'hawaii', 'hdmi', 'hear', 'help', 'hey', 'hi',
       'hot', 'http', 'immediately', 'instantly', 'insurance', 'iphone',
       'join', 'know', 'kyc', 'laptop', 'late', 'learning', 'lecture',
       'let', 'limited', 'loc

In [13]:
X_train_vec.toarray()

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.34074051],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.37641867]])

In [14]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

In [15]:
X_test_vec = vectorizer.transform(X_test)

In [16]:
from sklearn.svm import LinearSVC

In [17]:
model = LinearSVC()

In [18]:
model.fit(X_train_vec,y_train_enc )

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo rand

In [19]:
y_predict = model.predict(X_test_vec)

In [20]:
y_predict

array([1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1])

In [21]:
X_train_vec

<41x167 sparse matrix of type '<class 'numpy.float64'>'
	with 381 stored elements in Compressed Sparse Row format>

In [22]:
vectorizer.get_feature_names_out()

array(['000', '07045612389', '08098765432', '09012345678', '10', '15',
       '20', '200', '30am', '5000', '80090', '999', 'account', 'act',
       'afternoon', 'alert', 'amazon', 'amazongiftbonus', 'appreciate',
       'area', 'asap', 'assignment', 'atm', 'available', 'avoid',
       'awesome', 'bank', 'block', 'bring', 'bro', 'bug', 'buy', 'cable',
       'card', 'cash', 'catch', 'charger', 'chat', 'check', 'claim',
       'class', 'clean', 'click', 'code', 'coffee', 'com',
       'congratulation', 'consultation', 'crazyyyy', 'customer',
       'dataset', 'datingnow', 'datum', 'deactivate', 'deal', 'dear',
       'detail', 'dial', 'document', 'email', 'evening', 'exclusive',
       'expire', 'fi', 'final', 'forget', 'free', 'freegift', 'gift',
       'github', 'gym', 'hawaii', 'hdmi', 'hear', 'help', 'hey', 'hi',
       'hot', 'http', 'immediately', 'instantly', 'insurance', 'iphone',
       'join', 'know', 'kyc', 'laptop', 'late', 'learning', 'lecture',
       'let', 'limited', 'loc

In [23]:
X_train_vec.toarray()

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.34074051],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.37641867]])

In [24]:
vectorizer.idf_

array([4.04452244, 3.63905733, 3.63905733, 3.63905733, 3.63905733,
       3.63905733, 3.63905733, 3.63905733, 3.63905733, 4.04452244,
       4.04452244, 3.35137526, 3.35137526, 3.63905733, 3.63905733,
       3.63905733, 3.63905733, 3.63905733, 4.04452244, 4.04452244,
       3.63905733, 4.04452244, 4.04452244, 3.63905733, 4.04452244,
       3.63905733, 3.35137526, 4.04452244, 3.12823171, 3.63905733,
       3.63905733, 3.63905733, 3.63905733, 3.35137526, 4.04452244,
       3.63905733, 3.63905733, 4.04452244, 4.04452244, 3.35137526,
       3.63905733, 3.63905733, 3.35137526, 3.63905733, 3.63905733,
       2.43508453, 3.63905733, 3.63905733, 3.63905733, 4.04452244,
       3.63905733, 4.04452244, 3.35137526, 4.04452244, 3.63905733,
       4.04452244, 3.63905733, 4.04452244, 3.63905733, 4.04452244,
       3.63905733, 3.63905733, 3.63905733, 3.63905733, 3.63905733,
       3.63905733, 2.25276297, 3.63905733, 3.63905733, 3.63905733,
       3.63905733, 3.63905733, 3.63905733, 3.63905733, 4.04452